# Simple Linear Regression — One Model

This notebook builds only one Linear Regression model.

It includes:

- Basic dataset checks
- Correlation analysis
- Correlation heatmap
- Simple one-hot encoding
- VIF analysis
- Train/test split
- Model training
- Accuracy metrics
- Cross-validation
- Actual versus predicted comparison
- Residual analysis


## Step 1: Import libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from statsmodels.stats.outliers_influence import variance_inflation_factor


## Step 2: Load the dataset


In [ ]:
df = pd.read_csv("ecommerce_supervised_dataset.csv")

df.head()


## Step 3: Check the dataset


In [ ]:
print("Rows and columns:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())


## Step 4: Check data types


In [ ]:
df.dtypes


## Step 5: Check missing values


In [ ]:
df.isnull().sum()


## Step 6: View summary statistics


In [ ]:
df.describe()


# Correlation Analysis

Correlation is calculated only for numerical columns.


## Step 7: Select numerical columns


In [ ]:
numerical_data = df.select_dtypes(
    include=["int64", "float64"]
)

numerical_data.head()


## Step 8: Calculate the correlation matrix


In [ ]:
correlation_matrix = numerical_data.corr()

correlation_matrix


## Step 9: Check correlation with Purchase Amount


In [ ]:
correlation_with_target = correlation_matrix[
    "Purchase_Amount"
].sort_values(ascending=False)

correlation_with_target


## Step 10: Draw the correlation heatmap


In [ ]:
plt.figure(figsize=(11, 8))

plt.imshow(
    correlation_matrix,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.colorbar(label="Correlation")

plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=90
)

plt.yticks(
    range(len(correlation_matrix.index)),
    correlation_matrix.index
)

for row in range(len(correlation_matrix.index)):
    for column in range(len(correlation_matrix.columns)):
        plt.text(
            column,
            row,
            round(correlation_matrix.iloc[row, column], 2),
            ha="center",
            va="center",
            fontsize=8
        )

plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


# Prepare Data for One Model

The target is `Purchase_Amount`.

The classification column `High_Value_Customer` is removed because it is not an input for this regression model.


## Step 11: Separate input and target


In [ ]:
X = df.drop(
    columns=[
        "Purchase_Amount",
        "High_Value_Customer"
    ]
)

y = df["Purchase_Amount"]

print("Input shape:", X.shape)
print("Target shape:", y.shape)


## Step 12: View categorical columns


In [ ]:
categorical_columns = X.select_dtypes(
    include=["object"]
).columns.tolist()

categorical_columns


## Step 13: Apply simple one-hot encoding

`pd.get_dummies()` converts categorical columns into numerical dummy columns.

`drop_first=True` removes one category from each categorical variable.


In [ ]:
X_encoded = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

X_encoded.head()


## Step 14: Check the encoded data


In [ ]:
print("Encoded data shape:", X_encoded.shape)

print("\nEncoded columns:")
print(X_encoded.columns.tolist())


# VIF Analysis

VIF checks whether input features are highly related to each other.

General interpretation:

- VIF below 5: usually acceptable
- VIF above 5: possible multicollinearity
- VIF above 10: high multicollinearity


## Step 15: Calculate VIF


In [ ]:
vif_data = pd.DataFrame()

vif_data["Feature"] = X_encoded.columns

vif_data["VIF"] = [
    variance_inflation_factor(
        X_encoded.values,
        column_number
    )
    for column_number in range(
        X_encoded.shape[1]
    )
]

vif_data.sort_values(
    by="VIF",
    ascending=False
)


# Build the Linear Regression Model


## Step 16: Split training and testing data


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42
)

print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])


## Step 17: Create the model


In [ ]:
model = LinearRegression()


## Step 18: Train the model


In [ ]:
model.fit(
    X_train,
    y_train
)

print("Model training completed")


## Step 19: Make predictions


In [ ]:
predictions = model.predict(
    X_test
)

predictions[:10]


# Accuracy Check

For regression:

- Lower MAE, MSE, and RMSE are better.
- Higher R² is better.


## Step 20: Calculate model metrics


In [ ]:
mae = mean_absolute_error(
    y_test,
    predictions
)

mse = mean_squared_error(
    y_test,
    predictions
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test,
    predictions
)

print("Mean Absolute Error :", mae)
print("Mean Squared Error  :", mse)
print("Root Mean Squared Error:", rmse)
print("R² Score            :", r2)


# Model Validation

Five-fold cross-validation checks the model on five different data portions.


## Step 21: Perform cross-validation


In [ ]:
cv_scores = cross_val_score(
    LinearRegression(),
    X_encoded,
    y,
    cv=5,
    scoring="r2"
)

print("Cross-validation R² scores:")
print(cv_scores)

print("\nMean CV R²:", cv_scores.mean())
print("CV standard deviation:", cv_scores.std())


# Actual and Predicted Comparison


## Step 22: Create a comparison table


In [ ]:
results = pd.DataFrame({
    "Actual Purchase Amount": y_test.values,
    "Predicted Purchase Amount": predictions
})

results.head(10)


## Step 23: Plot actual versus predicted values


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    y_test,
    predictions
)

minimum_value = min(
    y_test.min(),
    predictions.min()
)

maximum_value = max(
    y_test.max(),
    predictions.max()
)

plt.plot(
    [minimum_value, maximum_value],
    [minimum_value, maximum_value],
    linestyle="--"
)

plt.xlabel("Actual Purchase Amount")
plt.ylabel("Predicted Purchase Amount")
plt.title("Actual vs Predicted Values")
plt.show()


# Residual Analysis

Residual = Actual value − Predicted value.


## Step 24: Calculate residuals


In [ ]:
residuals = y_test - predictions

residuals.head()


## Step 25: Draw the residual plot


In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    predictions,
    residuals
)

plt.axhline(
    y=0,
    linestyle="--"
)

plt.xlabel("Predicted Purchase Amount")
plt.ylabel("Residual")
plt.title("Residual Plot")
plt.show()


# Model Coefficients


## Step 26: Display feature coefficients


In [ ]:
coefficients = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Coefficient": model.coef_
})

coefficients.sort_values(
    by="Coefficient",
    ascending=False
)


## Step 27: Display final results


In [ ]:
final_results = pd.DataFrame({
    "Metric": [
        "MAE",
        "MSE",
        "RMSE",
        "Test R²",
        "Mean CV R²"
    ],
    "Value": [
        mae,
        mse,
        rmse,
        r2,
        cv_scores.mean()
    ]
})

final_results


## Conclusion

This notebook builds one Linear Regression model using numerical and one-hot encoded categorical features.

The model is evaluated using:

- MAE
- MSE
- RMSE
- Test R²
- Five-fold cross-validation R²
- Actual versus predicted values
- Residual analysis
